In [0]:
# ============================================
# NOTEBOOK 1: Bronze Layer - Raw Data Ingestion
# PROJECT: Wikipedia Clickstream Analytics
# ============================================

# Step 1: Verify files are uploaded correctly
print("Checking uploaded files...")
display(dbutils.fs.ls("/Volumes/workspace/default/wikidata/"))

Checking uploaded files...


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/wikidata/clickstream-enwiki-2024-10.tsv.gz,clickstream-enwiki-2024-10.tsv.gz,500995036,1777008634000
dbfs:/Volumes/workspace/default/wikidata/pageviews-20241101-010000.gz,pageviews-20241101-010000.gz,42432354,1777008570000


In [0]:
# ============================================
# BRONZE: Load Raw Clickstream Data
# ============================================

clickstream_path = "/Volumes/workspace/default/wikidata/clickstream-enwiki-2024-10.tsv.gz"

# Read the TSV file
bronze_clickstream = spark.read \
    .option("sep", "\t") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .csv(clickstream_path)

# Rename columns
bronze_clickstream = bronze_clickstream \
    .withColumnRenamed("_c0", "source_page") \
    .withColumnRenamed("_c1", "target_page") \
    .withColumnRenamed("_c2", "click_type") \
    .withColumnRenamed("_c3", "click_count")

# Add ingestion timestamp
from pyspark.sql.functions import current_timestamp
bronze_clickstream = bronze_clickstream.withColumn("ingested_at", current_timestamp())

print("Clickstream record count:", bronze_clickstream.count())
display(bronze_clickstream.limit(5))

Clickstream record count: 34170973


source_page,target_page,click_type,click_count,ingested_at
other-empty,Main_Page,external,120677075,2026-04-24T05:43:27.517Z
Chinese_water_dragon,Arthropod,link,12,2026-04-24T05:43:27.517Z
other-search,Lyle_and_Erik_Menendez,external,7756271,2026-04-24T05:43:27.517Z
Chinese_water_snake,Snake,link,12,2026-04-24T05:43:27.517Z
other-internal,Main_Page,external,7469673,2026-04-24T05:43:27.517Z


In [0]:
# ============================================
# BRONZE: Load Raw Pageview Data
# ============================================

pageview_path = "/Volumes/workspace/default/wikidata/pageviews-20241101-010000.gz"

# Read the pageview file
bronze_pageviews = spark.read \
    .option("sep", " ") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .csv(pageview_path)

# Rename columns
bronze_pageviews = bronze_pageviews \
    .withColumnRenamed("_c0", "language_code") \
    .withColumnRenamed("_c1", "article_title") \
    .withColumnRenamed("_c2", "view_count") \
    .withColumnRenamed("_c3", "response_size")

# Add ingestion timestamp
bronze_pageviews = bronze_pageviews.withColumn("ingested_at", current_timestamp())

print("Pageview record count:", bronze_pageviews.count())
display(bronze_pageviews.limit(5))

Pageview record count: 4977578


language_code,article_title,view_count,response_size,ingested_at
null,File:DHFR_methotrexate_inhibitor.svg,1,0,2026-04-24T05:47:32.877Z
null,File:Pahang_location_map.svg,1,0,2026-04-24T05:47:32.877Z
null,File:Wikifunctions-logo.svg,2,0,2026-04-24T05:47:32.877Z
null,Module:Documentation,1,0,2026-04-24T05:47:32.877Z
null,Special:MyLanguage/Wikifunctions:Report_a_technical_problem,2,0,2026-04-24T05:47:32.877Z


In [0]:
# ============================================
# BRONZE: Save to Delta Lake Tables
# ============================================

# Save clickstream
bronze_clickstream.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_clickstream")

print("✅ bronze_clickstream table saved!")

# Save pageviews
bronze_pageviews.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_pageviews")

print("✅ bronze_pageviews table saved!")
print("🎉 Bronze Layer Complete!")

✅ bronze_clickstream table saved!
✅ bronze_pageviews table saved!
🎉 Bronze Layer Complete!
